In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from allensdk.brain_observatory.behavior.behavior_project_cache import VisualBehaviorNeuropixelsProjectCache

cache_dir = "./allen_vbn_cache"
os.makedirs(cache_dir, exist_ok=True)

cache = VisualBehaviorNeuropixelsProjectCache.from_s3_cache(cache_dir=cache_dir)

processed_file = "./processed_vbn_session_1108334384.npz"

ecephys_sessions.csv: 100%|██████████| 64.4k/64.4k [00:00<00:00, 916kMB/s]
behavior_sessions.csv: 100%|██████████| 562k/562k [00:00<00:00, 3.56MMB/s] 
units.csv: 100%|██████████| 134M/134M [00:32<00:00, 4.06MMB/s]   
probes.csv: 100%|██████████| 130k/130k [00:00<00:00, 876kMB/s]  
channels.csv: 100%|██████████| 27.9M/27.9M [00:09<00:00, 3.01MMB/s]  


In [2]:
sessions = cache.get_ecephys_session_table()
print("Total sessions:", len(sessions))

units = cache.get_unit_table()
print("Total units:", len(units))

Total sessions: 103
Total units: 319013


In [3]:
target_regions = ['VISp', 'VISl', 'VISrl', 'VISam', 'VISpm', 'VISal', 'LGd', 'LP']
region_counts = units.loc[units['structure_acronym'].isin(target_regions), 'structure_acronym'].value_counts()
print(region_counts)
print("Total units in target regions:", region_counts.sum())

VISp     21567
VISpm    19772
VISl     19525
VISal    17442
VISam    16938
VISrl    15319
LP        5479
LGd       3866
Name: structure_acronym, dtype: int64
Total units in target regions: 119908


In [4]:
global_target_units = units[units['structure_acronym'].isin(target_regions)].copy()
session_region_counts = global_target_units.groupby(['ecephys_session_id', 'structure_acronym']).size().unstack(fill_value=0)

good_sessions = session_region_counts[
    (session_region_counts.get('VISp', 0) > 20) &
    (session_region_counts.get('LGd', 0) > 20)
]

print("Sessions meeting VISp/LGd threshold:", len(good_sessions))

Sessions meeting VISp/LGd threshold: 57


In [ ]:
selected_sessions = [1108334384, 1087723305, 1043752325]

session = cache.get_ecephys_session(ecephys_session_id=selected_sessions[0])

ecephys_session_1108334384.nwb:  30%|███       | 689M/2.26G [05:02<11:28, 2.29MMB/s]     

In [ ]:
trials = session.trials
print("Total trials:", len(trials))

In [ ]:
print(trials[['hit', 'miss', 'false_alarm', 'correct_reject']].sum())

In [ ]:
session_units = session.get_units()
print("Total units in this session:", len(session_units))

In [ ]:
unit_regions = units[['structure_acronym']].copy()
session_units_with_regions = session_units.join(unit_regions, how='left')
target_regions = ['VISp', 'VISl', 'VISrl', 'VISam', 'VISpm', 'VISal', 'LGd', 'LP']
session_units_filtered = session_units_with_regions[
    (session_units_with_regions['quality'] == 'good') &
    (session_units_with_regions['structure_acronym'].isin(target_regions))
].copy()

print(f"Good units in target regions: {len(session_units_filtered)}")
print(session_units_filtered['structure_acronym'].value_counts())

In [ ]:
change_trials = trials[(trials['hit'] == True) | (trials['miss'] == True)].copy()
change_trials['outcome'] = change_trials['hit'].map({True: 'hit', False: 'miss'})

print(f"Change trials: {len(change_trials)}")
print(change_trials['outcome'].value_counts())
print(change_trials[['start_time', 'change_time_no_display_delay', 'outcome']].head())

In [ ]:
bin_size = 0.05  
window_start = -1.0  
window_end = 1.0    
bins = np.arange(window_start, window_end + bin_size, bin_size)
n_bins = len(bins) - 1
spike_times = session.spike_times
unit_ids = session_units_filtered.index.values
change_times = change_trials['change_time_no_display_delay'].values
print(f"Building firing rate matrix: {len(change_trials)} trials x {len(unit_ids)} units x {n_bins} bins")

firing_rates = np.zeros((len(change_times), len(unit_ids), n_bins))

for i, ct in enumerate(change_times):
    if np.isnan(ct):
        continue
    for j, uid in enumerate(unit_ids):
        spikes = spike_times[uid]
       
        relative_spikes = spikes - ct
        counts, _ = np.histogram(relative_spikes, bins=bins)
        firing_rates[i, j, :] = counts / bin_size  

print(f"Firing rate matrix shape: {firing_rates.shape}")
print(f"(trials, units, time_bins)")

We aligned spike times to the stimulus change and converted them into firing rates using 50 ms bins from -1 to +1 seconds around the change.

In [ ]:
np.savez(
    processed_file,
    firing_rates=firing_rates,
    trial_labels=trial_labels,
    trial_labels_binary=trial_labels_binary,
    brain_regions=brain_regions,
    unit_ids=unit_ids,
    bin_centers=bin_centers,
    bin_size=np.array(bin_size),
)
print(f"Saved to {processed_file}")

In [ ]:
data = np.load(processed_file, allow_pickle=True)
print("Saved keys:", data.files)
print("firing_rates shape:", data["firing_rates"].shape)

## Some Graphs 

In [ ]:
trial_count_df = pd.Series(trial_labels).value_counts().reindex(["hit", "miss"])

plt.figure(figsize=(6, 4))
trial_count_df.plot(kind="bar")
plt.title("Hit vs Miss Trial Counts")
plt.xlabel("Trial Outcome")
plt.ylabel("Number of Trials")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
region_count_df = pd.Series(brain_regions).value_counts()

plt.figure(figsize=(8, 4))
region_count_df.plot(kind="bar")
plt.title("Filtered Units by Brain Region")
plt.xlabel("Brain Region")
plt.ylabel("Number of Units")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
hit_mask = trial_labels == "hit"
miss_mask = trial_labels == "miss"

hit_mean = firing_rates[hit_mask].mean(axis=(0, 1))
miss_mean = firing_rates[miss_mask].mean(axis=(0, 1))

plt.figure(figsize=(8, 5))
plt.plot(bin_centers, hit_mean, label="Hit")
plt.plot(bin_centers, miss_mean, label="Miss")
plt.axvline(0, linestyle="--")
plt.title("Average Population Firing Rate Over Time")
plt.xlabel("Time from Change (s)")
plt.ylabel("Average Firing Rate (Hz)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
c